# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

api_key = "ollama"    
MODEL = 'qwen3:14b'
openai = OpenAI(api_key=api_key, base_url="http://localhost:11434/v1")

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'courses/services', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'courses/services',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'courses/services', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog/news', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter/x', 'url': 'https://twitter.com/edwarddonner'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling qwen3:14b
Found 4 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling qwen3:14b
Found 9 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'brand guidelines', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'changelog', 'url': 'https://huggingface.co/changelog'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling qwen3:14b
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-M2.7
Updated
1 day ago
•
143k
•
848
tencent/HY-Embodied-0.5
Updated
3 days ago
•
1.06k
•
769
zai-org/GLM-5.1
Updated
about 17 hours ago
•
94.4k
•
1.28k
google/gemma-4-31B-it
Updated
6 days ago
•
3.2M
•
1.98k
Qwen/Qwen3.6-35B-A3B
Updated
1 day ago
•
422
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
492
OmniVoice
🌍
492
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.07k
Wan2.2 14B Preview
🐌
2.07k
generate a video from an image with a text prompt
Running
87
Bonsai 1-bit WebGPU
🌳
87
R

In [23]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:20_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling qwen3:14b
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-M2.7\nUpdated\n1 day ago\n•\n143k\n•\n848\ntencent/HY-Embodied-0.5\nUpdated\n3 days ago\n•\n1.06k\n•\n769\nzai-org/GLM-5.1\nUpdated\nabout 17 hours ago\n•\n94.4k\n•\n1.28k\ngoogle/gemma-4-31B-it\nUpdated\n6 days ago\n•\n3.2M\n•\n1.98k\nQwen/Qwen3.6-35B-A3B\nUpdated\n1 day ago\n•\n423\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n492\nOmniVoice\n🌍\n492

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling qwen3:14b
Found 3 relevant links


# Hugging Face: Building the Future of AI Together  

---

## **About Us**  
Hugging Face is the **collaboration platform for the machine learning community**, empowering engineers, scientists, and creators to build an open and ethical AI future. Our mission is to democratize AI by providing tools, resources, and a vibrant community where innovation thrives.  

At the heart of our platform is the **Hugging Face Hub**, a central space for sharing, discovering, and experimenting with open-source machine learning models, datasets, and applications. With millions of models and datasets available, we’re fostering a world where collaboration drives progress.  

---

## **What We Offer**  
### **For Developers & Researchers**  
- **Models**: Explore and contribute to **2M+ open-source models** (e.g., Qwen, Gemma, GLM) across text, image, audio, and 3D modalities.  
- **Datasets**: Access **500K+ datasets** for training and benchmarking, from language reasoning to mental health analysis.  
- **Spaces**: Build and deploy AI apps in **1M+ applications**, including voice cloning, image editing, and video generation.  
- **Buckets**: Store and manage large-scale data securely.  

### **For Enterprises**  
Scale your AI initiatives with **enterprise-grade security, access controls, and dedicated support**. Key features include:  
- **ZeroGPU** and advanced compute options for scalability.  
- **Private storage, audit logs, and resource groups** for secure collaboration.  
- **Inference Providers** with usage analytics and spending limits.  
- **Priority support** and **custom security policies** tailored to your organization.  

---

## **Our Culture**  
Hugging Face is driven by a **community-first ethos**, where open-source innovation and ethical AI are core values. We believe in:  
- **Collaboration**: Breaking down barriers between individuals and organizations.  
- **Innovation**: Pushing the edge of AI research with cutting-edge tools and libraries.  
- **Inclusivity**: Supporting 600+ languages and diverse use cases, from education to healthcare.  
- **Sustainability**: Promoting responsible AI development through transparent practices.  

---

## **Join Our Community**  
Whether you’re a developer, researcher, or enterprise, Hugging Face offers opportunities to:  
- **Contribute** to open-source projects and shape the future of AI.  
- **Learn** through tutorials, forums, and documentation.  
- **Connect** with a global network of ML enthusiasts and industry leaders.  

---

## **Careers**  
Hugging Face is a top destination for talent in AI and machine learning. We offer roles in engineering, research, product, and more—where you can work on **impactful projects** that drive the AI revolution. Explore opportunities at [Hugging Face Careers](#).  

---

## **Get Started**  
- **Individuals**: [Sign Up](#) and explore 2M+ models.  
- **Teams**: Subscribe to **Team plans** starting at **$20/user/month**.  
- **Enterprises**: [Contact sales](#) for tailored solutions.  

---  
**Hugging Face – The AI community building the future.**  
[Log In](#) | [Explore Models](#) | [Browse Datasets](#)  

---  
*Colors: #FFD21E, #FF9D00, #6B7280*  
*For brand assets, visit [Hugging Face Brand Universe](#).*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling qwen3:14b
Found 6 relevant links


# **Hugging Face: Empowering the AI Community**  
**Where Innovation Meets Collaboration**  

---

## **About Us**  
Hugging Face is the leading open-source community platform for artificial intelligence. We empower researchers, developers, and enterprises to build, share, and deploy cutting-edge AI models, datasets, and applications. With **2M+ models**, **1M+ Spaces**, and a thriving global community, we are redefining the future of machine learning.  

**Key Focus Areas:**  
- **Open-source collaboration** for accelerated innovation.  
- **Multimodal AI** (text, vision, audio, code, and beyond).  
- **Ethical AI** through transparency, accessibility, and responsible research.  

---

## **Our Platform: The Hub of AI Innovation**  
**Host, Share, and Deploy AI Everywhere**  
- **Models & Datasets**: Publish and discover state-of-the-art AI models (e.g., Transformers, Diffusers) and datasets for NLP, computer vision, audio, and more.  
- **Spaces**: Build and share interactive AI apps (e.g., chatbots, image generators) with **10× faster deployment** via **ZeroGPU**.  
- **Buckets**: Store and manage large files (models, datasets) with **per-TB pricing** and high-throughput transfers.  
- **Inference Providers**: Access **200K+ models** from 10+ partners for seamless deployment.  

**Key Tools & Libraries:**  
- **Transformers**: PyTorch & TensorFlow libraries for NLP, vision, and audio tasks.  
- **Datasets**: Easy access to 10K+ curated datasets.  
- **Tokenizers**: Fast, efficient tokenization for research and production.  
- **Optimum**: Optimize training/inference on AWS, Google Cloud, and more.  

---

## **Enterprise Solutions: Scalable AI for Business**  
**Tailored for Teams & Organizations**  
- **Team Plan** ($20/user/month):  
  - SSO (SAML/OIDC), audit logs, and granular access controls.  
  - Advanced compute options, private storage, and enterprise-grade security.  
- **Enterprise Plan** (Custom Pricing):  
  - SCIM provisioning, legal compliance, and dedicated support.  
  - Unlimited storage, API rate limits, and automated user management.  

**Why Choose Hugging Face for Enterprise?**  
- **Accelerate AI adoption** with pre-built tools and seamless cloud integration (AWS, Azure, Google Cloud).  
- **Reduce costs** with optimized storage (up to 25% off for 200TB+).  
- **Ensure security** with end-to-end encryption, audit trails, and role-based access.  

---

## **Community & Research: Driving AI Forward**  
**A Hub for Collaboration & Innovation**  
- **Blog & Articles**: Stay updated on research breakthroughs, tutorials, and case studies (e.g., *Darwin-27B-Opus*, *VAANI Dataset*, *Nucleus-Image*).  
- **Open Source Contributions**: From OCR models to emotion-driven TTS, our community shapes the future of AI.  
- **Partnerships**: Collaborate with leading organizations to advance AI ethics, multimodal models, and more.  

**Recent Highlights:**  
- **Multimodal Models**: *NEO-unify*, *BidirLM*, and *Holo3* push the boundaries of unified AI.  
- **Efficiency & Ethics**: Research on *KV Caching*, *RLHF*, and *ablation techniques* ensures scalable, responsible AI.  
- **Developer Tools**: *Easyaligner*, *LiteCoder*, and *mlx* simplify tasks from alignment to deployment.  

---

## **Careers: Join the AI Revolution**  
Hugging Face is hiring! Shape the future of AI by contributing to open-source projects, enterprise solutions, and research. Explore opportunities in engineering, research, product, and more.  
**Visit our [Careers Page](https://huggingface.co/careers) to apply.**  

---

## **Get Started Today**  
Whether you're a researcher, developer, or enterprise, Hugging Face provides the tools, community, and infrastructure to turn your AI ideas into reality.  

**Visit [huggingface.co](https://huggingface.co) to explore models, datasets, and Spaces.**  
**Join our [community](https://discuss.huggingface.co) and collaborate with thousands of AI enthusiasts.**  

---

**Hugging Face: Building the Future of AI, Together.** 🌐🤖

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>